# Week 15: Graph Neural Networks & Geometric Deep Learning
## Notebook 04: GraphSAGE & Neighborhood Mini-Batch Sampling

### 1. Conceptual Intuition & Real-World Use Case
Vanilla GCNs and GATs are fundamentally **transductive**:
- They require the entire graph adjacency matrix $A$ in memory during forward and backward passes.
- They cannot naturally infer embeddings for brand-new nodes added to the graph after training.
- They cannot scale to billion-node enterprise graphs (e.g. Pinterest PinSage with 3 billion pins and 18 billion edges).

In 2017, Hamilton et al. created **GraphSAGE (Sample and Aggregate)**:
1. **Inductive:** Learns aggregator functions that generalize to previously unseen nodes.
2. **Neighborhood Sampling:** Instead of aggregating all neighbors, it uniformly samples a **fixed number** of neighbors (e.g. $S_1 = 10$ at hop 1, $S_2 = 25$ at hop 2). This guarantees constant computational and memory bounds per mini-batch!

---

### 2. Mathematical Foundations
For a target node $v$ at search depth $k \in \{1, \dots, K\}$:

1. **Neighborhood Sampling:**
   $$\mathcal{N}_k(v) \sim \text{UniformSample}(\mathcal{N}(v), S_k)$$
   where $S_k$ is the sample size (typically $10$ to $25$).

2. **Aggregation:**
   $$h_{\mathcal{N}(v)}^{(k)} = \text{AGGREGATE}_k\left( \{ h_u^{(k-1)}, \forall u \in \mathcal{N}_k(v) \} \right)$$
   Supported aggregators:
   - **Mean Aggregator:** $h_{\mathcal{N}(v)} = \frac{1}{|\mathcal{N}_k(v)|} \sum_{u} h_u^{(k-1)}$
   - **Pooling Aggregator:** $\max(\sigma(W_{pool} h_u + b))$
   - **LSTM Aggregator:** Higher expressiveness by feeding shuffled neighbor sequences into an LSTM cell.

3. **Concatenation & Update:**
   $$h_v^{(k)} = \sigma\left( W^{(k)} \cdot [h_v^{(k-1)} \parallel h_{\mathcal{N}(v)}^{(k)}] \right)$$
   $$h_v^{(k)} \leftarrow \frac{h_v^{(k)}}{\|h_v^{(k)}\|_2} \quad \text{(L2 Normalization)}$$

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Simulate a large social graph with high-degree hubs
num_nodes = 20
feature_dim = 8

# Features for all nodes
node_features = np.random.randn(num_nodes, feature_dim)

# Adjacency list: Node 0 is a massive hub with 12 neighbors
graph_adj = {i: [] for i in range(num_nodes)}
edges = [
    (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9), (0, 10), (0, 11), (0, 12),
    (1, 13), (2, 14), (3, 15), (4, 16), (5, 17), (6, 18), (7, 19)
]
for u, v in edges:
    graph_adj[u].append(v)
    graph_adj[v].append(u)

print(f"Total graph nodes: {num_nodes}")
print(f"Node 0 total degree: {len(graph_adj[0])} (High-degree hub)")

# Step 2: Implement GraphSAGE Layer with Uniform Sampling
class GraphSAGELayer:
    def __init__(self, in_features, out_features, sample_size=4):
        self.sample_size = sample_size
        # Weight matrix multiplies concatenated [h_v || h_N(v)]
        self.W = np.random.randn(2 * in_features, out_features) * 0.3

    def sample_neighbors(self, node_id):
        neighbors = graph_adj[node_id]
        if len(neighbors) == 0:
            return [node_id]  # Fallback self
        # Uniform sampling with replacement if neighborhood is smaller than sample_size
        replace = len(neighbors) < self.sample_size
        return np.random.choice(neighbors, size=self.sample_size, replace=replace)

    def aggregate_mean(self, neighbor_features):
        return np.mean(neighbor_features, axis=0)

    def forward_node(self, target_node, current_features):
        h_v = current_features[target_node]
        
        # Sample neighbors
        sampled_nbrs = self.sample_neighbors(target_node)
        nbr_feats = current_features[sampled_nbrs]
        
        # Aggregate
        h_nbr = self.aggregate_mean(nbr_feats)
        
        # Concatenate self and neighbor representation
        h_cat = np.concatenate([h_v, h_nbr])
        
        # Linear projection + ReLU
        h_new = np.maximum(0, h_cat @ self.W)
        
        # L2 Normalization
        norm = np.linalg.norm(h_new)
        if norm > 0:
            h_new = h_new / norm
        return h_new, sampled_nbrs

# Step 3: Run GraphSAGE on mini-batch of target nodes [0, 1]
sage_layer = GraphSAGELayer(in_features=feature_dim, out_features=16, sample_size=4)

print("\nProcessing Target Node 0:")
h_0_new, sampled_0 = sage_layer.forward_node(0, node_features)
print(f"  Total neighbors: {len(graph_adj[0])}")
print(f"  Sampled neighbors (Fixed size 4): {sampled_0}")
print(f"  Output embedding norm: {np.linalg.norm(h_0_new):.4f}")
print("  Output embedding shape:", h_0_new.shape)

print("\nProcessing Target Node 1:")
h_1_new, sampled_1 = sage_layer.forward_node(1, node_features)
print(f"  Sampled neighbors: {sampled_1}")
print("  Output embedding shape:", h_1_new.shape)

### 3. Step-by-Step Code Walkthrough
1. **Neighborhood Sampler:** For hub node 0 with 12 connections, `np.random.choice` selects exactly 4 neighbors, bounding memory usage.
2. **Mean Aggregator:** Takes the average feature vector of sampled neighbors.
3. **Self-Concatenation:** $[h_v \parallel h_{\mathcal{N}(v)}]$ preserves ego-node identity, preventing feature dilution.
4. **L2 Normalization:** Normalizing projected representations onto the unit hypersphere stabilizes training and makes cosine distance calculations mathematically sound.

---

### 4. Industry Applications & Trade-offs
- **Pinterest PinSage:** Powers recommendations for billions of images and search queries by streaming random walks into GraphSAGE mini-batches.
- **E-Commerce Real-time Fraud:** When a new user signs up, their embedding is computed immediately from their 1-hop and 2-hop transaction partners without retraining the model.

---

### 5. Top Interview Questions & Answers
**Q1: What is the primary difference between transductive and inductive learning in GNNs?**
> **Answer:** Transductive models (e.g. GCN) require all graph nodes and edges during training and cannot make predictions on nodes not present in the original graph without full re-training. Inductive models (e.g. GraphSAGE) learn generalized aggregation rules parameterized by weights $W$, allowing zero-shot embedding of entirely new nodes.

**Q2: What is the "neighbor explosion" problem during multi-hop aggregation?**
> **Answer:** If average node degree is $d$, a 2-hop neighborhood expands by $d^2$, and a 3-hop neighborhood by $d^3$. On real scale-free web graphs with hubs ($d > 1000$), un-sampled aggregation causes memory and latency to explode. Fixed-size sampling ($S_1 \times S_2$) ensures bounded mini-batch computation ($O(\prod S_k)$).